# Nifty 50 Sector Rotation & Anomaly Detection
## Notebook 1 — Data Pull
**Author:** Prathmesh Joshi  
**Source:** Live data via Yahoo Finance API (yfinance)  
**Period:** January 2021 – January 2024  
**Stocks:** 15 Nifty 50 large-cap stocks across 5 sectors

In [ ]:
# Install required libraries if not already installed
# Run this cell once
# !pip install yfinance pandas

In [ ]:
import yfinance as yf
import pandas as pd
import os

print('Libraries loaded successfully')

In [ ]:
# Define 15 stocks across 5 sectors
# Using NSE tickers — .NS suffix required for Indian stocks on Yahoo Finance
tickers = {
    'IT':      ['TCS.NS', 'INFY.NS', 'WIPRO.NS'],
    'Banking': ['HDFCBANK.NS', 'ICICIBANK.NS', 'SBIN.NS'],
    'Energy':  ['RELIANCE.NS', 'ONGC.NS', 'POWERGRID.NS'],
    'FMCG':    ['HINDUNILVR.NS', 'NESTLEIND.NS', 'BRITANNIA.NS'],
    'Auto':    ['MARUTI.NS', 'TATAMOTORS.NS', 'BAJAJ-AUTO.NS']
}

print('Tickers defined:', sum(len(v) for v in tickers.values()), 'stocks across', len(tickers), 'sectors')

In [ ]:
# Pull 3 years of daily OHLCV data for each stock
# auto_adjust=True gives clean adjusted prices accounting for splits and dividends
all_data = []

for sector, stocks in tickers.items():
    for ticker in stocks:
        print(f'Downloading {ticker} ({sector})...')
        df = yf.download(
            ticker,
            start='2021-01-01',
            end='2024-01-01',
            auto_adjust=True,
            progress=False
        )
        # Tag each row with stock and sector info
        df['Ticker'] = ticker
        df['Sector'] = sector
        df = df.reset_index()
        # Flatten multi-level columns if present
        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
        all_data.append(df)

print('\nAll downloads complete.')

In [ ]:
# Combine all stocks into one dataframe
raw_df = pd.concat(all_data, ignore_index=True)

# Keep only the columns we need
raw_df = raw_df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker', 'Sector']]

print('Total rows:', len(raw_df))
print('Date range:', raw_df['Date'].min(), 'to', raw_df['Date'].max())
print('\nStocks per sector:')
print(raw_df.groupby('Sector')['Ticker'].nunique())

In [ ]:
# Quick data quality check — look for missing values
print('Missing values per column:')
print(raw_df.isnull().sum())
print('\nSample data:')
raw_df.head(10)

In [ ]:
# Save raw data to CSV
# This file will be loaded into MySQL and used in Notebook 2
os.makedirs('../data', exist_ok=True)
raw_df.to_csv('../data/nifty_raw.csv', index=False)
print('Raw data saved to ../data/nifty_raw.csv')
print('Rows saved:', len(raw_df))